<a href="https://colab.research.google.com/github/chakradhar875/Agentic-AI/blob/main/Experiment4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q -U langchain langchain-community langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.0/147.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.1/565.1 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.9/248.9 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [4]:
import sqlite3

# Create SQLite database
conn = sqlite3.connect("students.db")
cursor = conn.cursor()

# Create students table
cursor.execute("""
CREATE TABLE IF NOT EXISTS students (
    id INTEGER PRIMARY KEY,
    name TEXT,
    branch TEXT,
    marks INTEGER
)
""")

# Insert sample student data
students = [
    (1, "Anil", "CSE", 85),
    (2, "Priya", "CSE", 92),
    (3, "Rahul", "ECE", 78),
    (4, "Sneha", "CSE", 88),
    (5, "Kiran", "ECE", 95)
]

cursor.executemany(
    "INSERT OR IGNORE INTO students VALUES (?, ?, ?, ?)",
    students
)

conn.commit()

print("SQLite database created successfully!")
print("Students table created successfully!")

SQLite database created successfully!
Students table created successfully!


In [5]:
import sqlite3

conn = sqlite3.connect("students.db")
cursor = conn.cursor()

cursor.execute("SELECT * FROM students")

rows = cursor.fetchall()

print("Students in database:\n")

for row in rows:
    print(row)

Students in database:

(1, 'Anil', 'CSE', 85)
(2, 'Priya', 'CSE', 92)
(3, 'Rahul', 'ECE', 78)
(4, 'Sneha', 'CSE', 88)
(5, 'Kiran', 'ECE', 95)


In [6]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///students.db")

print("Database connected to LangChain successfully!")
print("Available tables:", db.get_usable_table_names())

/tmp/ipykernel_1481/1968738485.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


Database connected to LangChain successfully!
Available tables: ['students']


In [7]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Enter your Gemini API key: ")

Enter your Gemini API key: ··········


In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash"
)

print("Gemini connected successfully!")

Gemini connected successfully!


In [9]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(
    db=db,
    llm=llm
)

tools = toolkit.get_tools()

print("SQL tools created successfully!")
print("\nAvailable tools:")

for tool in tools:
    print("-", tool.name)

SQL tools created successfully!

Available tools:
- sql_db_query
- sql_db_schema
- sql_db_list_tables
- sql_db_query_checker


In [10]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""You are a SQL database assistant.

Use the available SQL tools to answer questions about the students database.

Always inspect the database before answering.
Do not invent information.
Give a clear and concise final answer."""
)

print("SQL Agent created successfully!")

SQL Agent created successfully!


In [11]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "How many students are in the database?"
        }
    ]
})

print(response["messages"][-1].content)

[{'type': 'text', 'text': 'There are 5 students in the database.', 'extras': {'signature': 'EsABCr0BARFNMg84jOZEAMoD2hmmqG3D11YNFGW6uCdezpJxIYpFdm4pbO9ZEWcTgqowBQwtOXrMPHg3DEx3iv9tR+QYF6+tW6XtvFwjCDJL1iD9W+PBIcO5igsymtilG7piTG2GAr1OvEiCLgR5zw4D+U4X+JmgTe/70U00OB93ezXrQrrSx1/rEJbQnQ5TmGQCjomMZkNEhPZW7Fz6tJtH7jwUOYg5g8/2tLbGaBO6WeZUU2514/m0FFrrvNOdm0mV'}}]


In [12]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Which student has the highest marks?"
        }
    ]
})

print(response["messages"][-1].content)

[{'type': 'text', 'text': 'The student with the highest marks is **Kiran** (Branch: ECE) with **95** marks.', 'extras': {'signature': 'EpECCo4CARFNMg/izDJtftJul8vE5meHMi68i+HaQb5fj9bHvhRBPzQUtBGjROIqaexMeW2R7bPr9nuZKikcnB9j8DKTjFiSeYpSDbgw/8TEoMPDSnODbIUj0CiHTZil50HcwLpSk9wiuu1iQ1VzWFTf1XKg4Hg5aMczOJhuqq/yk6YbzJ24iPQmvtYMiq83bh5DDi1QfpAZ+tuObS/sRa12SHjS99PcT33I6Fh3j8Qx7ocRHdXC6zUc3v420EMD4gjjYiuKo+EDlv43k9AfLXT7qHPGM5m1cpYGaiu3a6zS/mAAi+tpf3oqsBh6yXeCg07zBoVBNmMUeoAu2Y0VUT4kyR/8z8r6wz/Cfv0YeGhrIJN6'}}]


In [13]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "List all students who are in the CSE branch."
        }
    ]
})

print(response["messages"][-1].content)

[{'type': 'text', 'text': 'The students in the CSE branch are:\n\n1. **Anil** (ID: 1, Marks: 85)\n2. **Priya** (ID: 2, Marks: 92)\n3. **Sneha** (ID: 4, Marks: 88)', 'extras': {'signature': 'Eq0CCqoCARFNMg98W0Q7NSDK7rBzcRmMEhTrKCjf4yrPNm2z/2zfu8r8OpbNqwNsUKjBzWXacK9FLWfI+7XZH0NAg5xKp5k73npaqhtALg/5cUBAY4GxaRX/h9/BBMHg1Fl7IvJGnCqoBWNVS9PPp+9aWOKx7dSh/nYPXLNk3/zq5HsDpQG/xm/dlIENOIccsZw/Bk/mDLymgbIIIhJ7uFCoZvRtZCpejRWoYd4VcTLEXAY6rSBAyFb9N6hh0lFrXy8fU7pH1Lv72dMW+XknsC9scl4K5popAXjyZnqFIMY5g+ckBvI2X5RbjtNlwLrF+zYe31di/WSPS/KNA82u25a2IPdA0nCt+86bc9gApYWHyUY3R007QeG29uxks5KtlsX5pv9/UIMju2ybO2N86w=='}}]
